<a href="https://colab.research.google.com/github/lee-woonju/study-hongong-mldl/blob/main/07_2_%EC%8B%AC%EC%B8%B5%EC%8B%A0%EA%B2%BD%EB%A7%9D_%ED%8C%8C%EC%9D%B4%ED%86%A0%EC%B9%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 파이토치로 신경망 모델 만들기

## torchvision으로 데이터셋 불러오기

In [2]:
from torchvision.datasets import FashionMNIST

# root : 다운로드된 데이터 저장 위치
# train : True 훈련데이터, False 테스트데이터
# download : True 원격 데이터를 로컬에 저장 (default = False)
fm_train = FashionMNIST(root='.', train=True, download=True)
fm_test = FashionMNIST(root='.', train=False, download=True)

# 파이토치 텐서
type(fm_train.data)

# 타깃이 1차원 배열 = 정숫값 != 원핫인코딩

torch.Tensor

## 파이토치 텐서
- 쉽게 말해 **인공지능(딥러닝) 연산을 위해 만든 다차원 배열(숫자 묶음)**
- 넘파이 배열은 CPU에서만 작동, 파이토치 텐서는 GPU 계산 가능


### 원핫인코딩이 뭐였지?
범주형 데이터를 0과 1로 이루어진 이진 벡터로 변환하는 기술

In [4]:
print(fm_train.data.shape, fm_test.data.shape)
print(fm_train.targets.shape, fm_test.targets.shape)

torch.Size([60000, 28, 28]) torch.Size([10000, 28, 28])
torch.Size([60000]) torch.Size([10000])


In [5]:
train_input = fm_train.data
train_target = fm_train.targets

In [6]:
train_scaled = train_input/255.0

# 훈련세트를 다시 훈련세트와 검증세트로 나누기
from sklearn.model_selection import train_test_split

train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size=0.2, random_state=42)

# 훈련 세트와 검증 세트의 크기 확인
print(train_scaled.shape, val_scaled.shape)

torch.Size([48000, 28, 28]) torch.Size([12000, 28, 28])


In [8]:
# 파이토치의 층은 torch.nn 모듈 아래 위치
import torch.nn as nn

model = nn.Sequential(
    nn.Flatten(),         # 일렬로 펼치기. 2차원 이미지를 1차원 데이터로 편다.
    nn.Linear(784, 100),  # 첫 번째 은닉층. 입력된 784개의 숫자를 100개의 특징으로 압축(변환)
    nn.ReLU(),            # 활성화 함수. 인공신경망에 '비선형성'을 추가해주는 필터. 양수는 그대로 음수는 0으로
    nn.Linear(100, 10)    # 출력층. 최종 예측 값을 내보내는 측. 앞서 가공된 100개의 특징을 최종 10개의 숫자로 변환
)


### Linear
- 케라스의 Dense()와 같은 역할

#### 하이퍼파라미터
- 성능을 높이기 위해 **사람이 이리저리 바꿔볼 수 있는 값**
- 첫번째 nn.Linear(784, 100)에서 100은 특징을 몇개로 압축할지 튜닝 = 하이퍼파라미터
- 두번째 nn.Linear(100, 10)에서 10은 정답의 종류 = 하이퍼파라미터가 아님

### 렐루 함수
- 이전 층에서 계산되어 넘어온 복잡한 숫자들을 **쓸모 있는 값만 남기고 필터링하는 장치**
- 음수(-)는 무조건 0으로 만들고
- 양수(+)는 그대로 통과시킨다
- 기울기 소실 방지 : 시그모이드는 입력값이 커지면 기울기(미분값)가 0에 가까워져 학습이 멈춤

In [10]:
# 케라스의 summary() 메서드가 없음
!pip install torchinfo

In [11]:
from torchinfo import summary
summary(model, input_size=(32, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [32, 10]                  --
├─Flatten: 1-1                           [32, 784]                 --
├─Linear: 1-2                            [32, 100]                 78,500
├─ReLU: 1-3                              [32, 100]                 --
├─Linear: 1-4                            [32, 10]                  1,010
Total params: 79,510
Trainable params: 79,510
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 2.54
Input size (MB): 0.10
Forward/backward pass size (MB): 0.03
Params size (MB): 0.32
Estimated Total Size (MB): 0.45

> 파이토치는 명시적으로 GPU로 모델을 이동해야 함(케라스는 알아서 GPU에서 훈련)

In [12]:
import torch

# 내 컴퓨터에 NVIDIA 그래픽카드(GPU)가 있고, CUDA가 깔려있나 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=10, bias=True)
)

### cuda
- cuda == gpu ? **아님**
> torch.device("cuda" if torch.cuda.is_available() else "cpu")
- 왜 "gpu" 가 아니라 "cuda"일까 ?

#### gpu
- 계산을 엄청 잘하는 기계엔진 그 자체. 하드웨어

#### cuda
- "GPU 일 시키는 프로그래밍 도구". 소프트웨어
- NVIDIA 그래픽카드(GPU)가 탑재된 컴퓨터에서만 CUDA 사용 가능

> CUDA라는 드라이버(통로)를 통해 저 GPU 하드웨어에 데이터를 집어넣어라 명령하는 것

## 손실함수와 옵티마이저
- 손실함수 : 모델이 얼마나 못했는지 측정하는 채점 기준
- 옵티마이저 : 가중치를 수정해 나가는 '최적화 알고리즘'


In [14]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# model.parameters()
#

for params in model.parameters():
    print(params.shape)

torch.Size([100, 784])
torch.Size([100])
torch.Size([10, 100])
torch.Size([10])


## 모델 훈련
- 케라스의 fit() 메서드가 없음
- 에포크와 배치 경사 하강법을 위한 for문 직접 구현

In [15]:

epochs = 5
batches = int(len(train_scaled)/32)

for epoch in range(epochs): # 에포크 반복
  model.train()
  train_loss = 0  # 에포크 손실 초기화
  for i in range(batches):  # 배치 반복. 배치 = 1에포크에서 학습하는 최소 단위
    inputs = train_scaled[i*32:(i+1)*32].to(device)  # 배치 입력 준비
    targets = train_target[i*32:(i+1)*32].to(device)  # 배치 타겟 준비
    optimizer.zero_grad() # 이전 배치의 기울기가 남아있지 않게 청소(옵티마이저 그레디언트 초기화)
    outputs = model(inputs) # 모델에 입력 전달하여 예측값 계산
    loss = criterion(outputs, targets)  # 예측값과 정답 비교하여 손실 채점
    loss.backward()   # 손실 역전파. 가중치들의 기울기 계산
    optimizer.step()  # 모델 파라미터 업데이트. 옵티마이저를 통해 가중치 업데이트
    train_loss += loss.item() # 에포크 손실 기록
  print(f"에포크:{epoch+1}, 손실 : {train_loss/batches:.4f}") # 에포크 솔실 출력

에포크:1, 손실 : 0.5520
에포크:2, 손실 : 0.4071
에포크:3, 손실 : 0.3630
에포크:4, 손실 : 0.3354
에포크:5, 손실 : 0.3153
